# Подготовка данных и создание прототипа СППР

Команда «Движуха в кадрах».

В этом ноутбуке мы:
1. проверяем пропуски в таблицах «Образование», «ПК/ПП» и «Школы»;
2. проверяем в них выбросы;
3. готовим таблицу для прототипа;
4. строим интерактивные графики для двух пользовательских требований.

Требования взяты из нашего документа «Требования для разработки СППР»:

- **Директор школы** хочет сравнить кадровые показатели своей школы (возраст, квалификационные категории) со средними по городу.
- **Сотрудник городского органа управления образованием** хочет видеть сводные данные о педагогах: численность по округам и возрастную структуру.

In [1]:
import pandas as pd
import plotly.express as px

## 1. Загрузка данных

В файлах «Образование», «ПК-ПП» и «Педагоги» три листа: 2022, 2023 и 2024 год. Читаем каждый лист, добавляем столбец с годом и склеиваем всё в одну таблицу. В файле «Школы» лист один.

In [2]:
def load_all_years(file_name):
    parts = []
    for year in ['2022', '2023', '2024']:
        df = pd.read_excel('../data/' + file_name, sheet_name=year)
        df['year'] = int(year)  # запоминаем, с какого листа строка
        parts.append(df)
    return pd.concat(parts, ignore_index=True)

df_edu = load_all_years('Образование.xlsx')
df_pk = load_all_years('ПК-ПП.xlsx')
df_ped = load_all_years('Педагоги.xlsx')
df_school = pd.read_excel('../data/Школы.xlsx')

У таблицы «ПК/ПП» очень длинные названия столбцов, переименуем их, чтобы было удобнее писать код.

In [3]:
df_pk = df_pk.rename(columns={
    'Программа профессиональной переподготовки(ПП)/повышения квалификации(ПК)': 'Тип программы',
    'Дата получения документа/освоения программы': 'Год документа',
})

In [4]:
print('Образование:', df_edu.shape)
print('ПК/ПП:', df_pk.shape)
print('Школы:', df_school.shape)
print('Педагоги:', df_ped.shape)

Образование: (302719, 7)
ПК/ПП: (65733, 6)
Школы: (266, 4)
Педагоги: (428140, 16)


In [5]:
df_edu.head(3)

,id,Тип образования,Наименование учебного заведения,Дата окончания,Специализация,Квалификация,year
0,f583c34a49,"Высшее образование - специалитет, магистратура","ФГОУ ВПО ""Государственный технологический унив...",2006.0,Теория и методика преподавания иностранных язы...,"Лингвист, преподаватель",2022
1,08ff701e26,Высшее образование,Бийский государственный педагогический институт,1989.0,Физика,учитель физики и математики,2022
2,b813df019b,Высшее образование,МОПИ им. Н. К. Крупской,1993.0,Математика,Учитель математики,2022


In [6]:
df_pk.head(3)

,id,Наименование программы,Тип программы,Количество часов,Год документа,year
0,18ac5986cd,Методические подходы в подготовке к государств...,ПК,16,2022,2022
1,18ac5986cd,Организация образовательного процесса в услови...,ПК,22,2022,2022
2,18ac5986cd,Подготовка организаторов пунктов проведения эк...,ПК,16,2022,2022


In [7]:
df_school.head(3)

,Школа,Округ,Кол-во зданий,Общая численность контингента
0,325,Инновационный округ,6,1294
1,397,Инновационный округ,3,842
2,192,Северный округ,13,4505


### Дубликаты

Проверим, есть ли в таблицах полностью одинаковые строки.

In [8]:
print('Дубликатов в «Образовании»:', df_edu.duplicated().sum())
print('Дубликатов в «ПК/ПП»:', df_pk.duplicated().sum())
print('Дубликатов в «Школах»:', df_school.duplicated().sum())

Дубликатов в «Образовании»: 68349
Дубликатов в «ПК/ПП»: 12
Дубликатов в «Школах»: 0


Одинаковые строки не дают новой информации, а только завышают количество (например, число пройденных программ). Поэтому удаляем их.

In [9]:
df_edu = df_edu.drop_duplicates()
df_pk = df_pk.drop_duplicates()

print('Образование:', df_edu.shape)
print('ПК/ПП:', df_pk.shape)

Образование: (234370, 7)
ПК/ПП: (65721, 6)


## 2. Пропуски

Посчитаем количество пропусков в каждом столбце.

In [10]:
print('--- ПК/ПП ---')
print(df_pk.isna().sum())
print()
print('--- Школы ---')
print(df_school.isna().sum())

--- ПК/ПП ---
id                        0
Наименование программы    0
Тип программы             0
Количество часов          0
Год документа             0
year                      0
dtype: int64

--- Школы ---
Школа                            0
Округ                            0
Кол-во зданий                    0
Общая численность контингента    0
dtype: int64


В таблицах «ПК/ПП» и «Школы» пропусков нет, заполнять ничего не нужно.

Теперь «Образование». Для него выведем ещё и долю пропусков в процентах.

In [11]:
missing = df_edu.isna().sum()
missing_pct = (missing / len(df_edu) * 100).round(2)
pd.DataFrame({'пропусков': missing, '%': missing_pct})

,пропусков,%
id,0,0.00
Тип образования,0,0.00
Наименование учебного заведения,70,0.03
Дата окончания,649,0.28
Специализация,120683,51.49
Квалификация,155019,66.14
year,0,0.00


Пропуски есть в четырёх столбцах. Больше всего их в «Специализации» и «Квалификации».

При просмотре таблицы мы заметили, что в столбце «Наименование учебного заведения» часто стоит прочерк `-`. По смыслу это тоже пропуск, просто записанный по-другому. Посчитаем такие значения.

In [12]:
print('Прочерков:', (df_edu['Наименование учебного заведения'] == '-').sum())

Прочерков: 166875


Прочерков очень много. Заменим их на настоящие пропуски и пересчитаем таблицу.

In [13]:
df_edu.loc[df_edu['Наименование учебного заведения'] == '-', 'Наименование учебного заведения'] = None

missing = df_edu.isna().sum()
missing_pct = (missing / len(df_edu) * 100).round(2)
pd.DataFrame({'пропусков': missing, '%': missing_pct})

,пропусков,%
id,0,0.00
Тип образования,0,0.00
Наименование учебного заведения,166945,71.23
Дата окончания,649,0.28
Специализация,120683,51.49
Квалификация,155019,66.14
year,0,0.00


Чтобы выбрать способ заполнения, посмотрим, у каких типов образования пропусков больше всего. Для этого сделаем вспомогательные столбцы: `True`, если значение пропущено. Среднее по такому столбцу - это доля пропусков.

In [14]:
df_edu['нет специализации'] = df_edu['Специализация'].isna()
df_edu['нет квалификации'] = df_edu['Квалификация'].isna()
df_edu['нет заведения'] = df_edu['Наименование учебного заведения'].isna()
df_edu['нет даты'] = df_edu['Дата окончания'].isna()

helper_cols = ['нет специализации', 'нет квалификации', 'нет заведения', 'нет даты']
na_by_type = (df_edu.groupby('Тип образования')[helper_cols].mean() * 100).round(1)
na_by_type['записей'] = df_edu['Тип образования'].value_counts()
na_by_type.sort_values('записей', ascending=False)

,нет специализации,нет квалификации,нет заведения,нет даты,записей
Тип образования,,,,,
Дополнительное профессиональное образование,71.5,92.2,100.0,0.2,166875
Высшее образование,1.3,0.9,0.0,0.5,44747
"Высшее образование - специалитет, магистратура",1.2,0.9,0.1,0.1,8169
Среднее профессиональное образование,1.3,0.7,0.0,0.2,7993
Высшее образование - бакалавриат,1.4,1.9,0.1,0.1,5858
Среднее (полное) общее образование,99.1,98.5,12.2,3.5,343
Аспирантура,15.7,5.6,0.0,0.0,108
Среднее общее образование,91.8,91.8,2.0,0.0,98
Неполное высшее образование,13.6,40.9,0.0,0.0,66


In [15]:
# вспомогательные столбцы больше не нужны
df_edu = df_edu.drop(columns=helper_cols)

**Что видно из таблицы**

- Почти все пропуски приходятся на «Дополнительное профессиональное образование» (это курсы). У курсов нет специализации и квалификации в том смысле, как у диплома вуза, поэтому поля пустые. Название заведения у курсов не указано вообще ни разу.
- У высшего и среднего профессионального образования пропусков всего 1-2 %.
- У общего образования (школа) специализации и квалификации тоже нет, и это логично.

**Как заполняем и почему**

- **«Специализация», «Квалификация», «Наименование учебного заведения»** - заполняем словом «не указано».
  - Это текстовые столбцы, среднее или медиану для них посчитать нельзя.
  - Самое частое значение (мода) не подходит: тогда больше половины записей получили бы одну и ту же специальность, которой у них на самом деле нет.
  - Удалять строки нельзя: мы потеряем больше половины таблицы.
  - Удалять столбцы тоже не хочется: для высшего образования они почти полностью заполнены и пригодятся для фильтров.
- **«Дата окончания»** - заполняем медианой, но отдельно для каждого типа образования.
  - Пропусков мало (меньше 1 %), поэтому заполнение почти не повлияет на результат.
  - Год окончания сильно зависит от типа образования, поэтому общая медиана подошла бы плохо. Ниже это видно по таблице.
  - Берём медиану, а не среднее, потому что в данных есть очень старые годы, которые сдвигают среднее.

In [16]:
df_edu.groupby('Тип образования')['Дата окончания'].median()

Тип образования
Аспирантура                                                   2019.0
Высшее образование                                            2001.0
Высшее образование - бакалавриат                              2019.0
Высшее образование - подготовка кадров высшей квалификации    2014.0
Высшее образование - специалитет, магистратура                2007.0
Дополнительное профессиональное образование                   2018.0
Неполное высшее образование                                   2020.0
Основное общее образование                                    2013.0
Среднее (полное) общее образование                            2011.0
Среднее общее образование                                     2017.0
Среднее профессиональное образование                          1997.0
Name: Дата окончания, dtype: float64

In [17]:
# текстовые столбцы
df_edu['Специализация'] = df_edu['Специализация'].fillna('не указано')
df_edu['Квалификация'] = df_edu['Квалификация'].fillna('не указано')
df_edu['Наименование учебного заведения'] = df_edu['Наименование учебного заведения'].fillna('не указано')

# год окончания: для каждого типа образования считаем свою медиану
for edu_type in df_edu['Тип образования'].unique():
    this_type = df_edu['Тип образования'] == edu_type
    median_year = df_edu.loc[this_type, 'Дата окончания'].median()
    df_edu.loc[this_type & df_edu['Дата окончания'].isna(), 'Дата окончания'] = median_year

df_edu['Дата окончания'] = df_edu['Дата окончания'].astype(int)

print('Пропусков осталось:', df_edu.isna().sum().sum())

Пропусков осталось: 0


## 3. Выбросы

Выбросы ищем по правилу межквартильного размаха (IQR), как на семинаре: выбросом считается значение меньше `Q1 - 1.5·IQR` или больше `Q3 + 1.5·IQR`.

Проверяем числовые столбцы:
- «ПК/ПП» - количество часов;
- «Образование» - дата окончания;
- «Школы» - количество зданий и численность контингента.

Номер школы не проверяем, потому что это идентификатор, а не измерение.

In [18]:
def detect_outliers_iqr(df, column):
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    outliers = df[(df[column] < lower_bound) | (df[column] > upper_bound)]

    print(column)
    print(f'Границы: [{lower_bound:.2f}, {upper_bound:.2f}]')
    print(f'Найдено выбросов: {len(outliers)} ({len(outliers) / len(df) * 100:.2f}%)')
    print()

    return outliers

out_hours = detect_outliers_iqr(df_pk, 'Количество часов')
out_date = detect_outliers_iqr(df_edu, 'Дата окончания')
out_buildings = detect_outliers_iqr(df_school, 'Кол-во зданий')
out_pupils = detect_outliers_iqr(df_school, 'Общая численность контингента')

Количество часов
Границы: [4.00, 36.00]
Найдено выбросов: 3789 (5.77%)

Дата окончания
Границы: [1995.00, 2035.00]
Найдено выбросов: 20358 (8.69%)

Кол-во зданий
Границы: [-8.50, 27.50]
Найдено выбросов: 3 (1.13%)

Общая численность контингента
Границы: [-1618.62, 6864.38]
Найдено выбросов: 1 (0.38%)



Выбросы нашлись во всех четырёх столбцах. Разберём каждый по очереди.

### 3.1. ПК/ПП - количество часов

In [19]:
fig = px.box(df_pk, x='Количество часов', title='ПК/ПП: количество часов')
fig.show()

In [20]:
df_pk.groupby('Тип программы')['Количество часов'].describe()

,count,mean,std,min,25%,50%,75%,max
Тип программы,,,,,,,,
ПК,65658.0,23.106522,11.363687,16.0,16.0,18.0,24.0,182.0
ПП,63.0,260.000000,0.000000,260.0,260.0,260.0,260.0,260.0


Обычный курс повышения квалификации длится 16-24 часа, поэтому всё, что больше 36 часов, попало в выбросы. Но это не ошибки:

- все программы переподготовки (ПП) длятся ровно 260 часов - это их обычная длительность;
- у повышения квалификации (ПК) тоже бывают длинные курсы: 36, 72 и даже 182 часа.

**Решение:** исходные значения не удаляем и не меняем, потому что это реальные данные. Если удалить выбросы, из таблицы пропадёт вся переподготовка.

Но для графиков такие редкие большие значения неудобны: шкала растягивается, и основную массу плохо видно. Поэтому добавим отдельный столбец, где часы ограничены 99-м перцентилем (значения выше «прижимаются» к границе, как на семинаре).

In [21]:
limit = df_pk['Количество часов'].quantile(0.99)
df_pk['Часы (ограничено)'] = df_pk['Количество часов'].clip(upper=limit)

print('Граница (99-й перцентиль):', limit)
print('Сколько значений изменилось:', (df_pk['Количество часов'] > limit).sum())

Граница (99-й перцентиль): 72.0
Сколько значений изменилось: 125


### 3.2. Образование - дата окончания

Посмотрим, сколько записей приходится на каждый год окончания.

In [22]:
year_counts = df_edu['Дата окончания'].value_counts().sort_index()

fig = px.bar(x=year_counts.index, y=year_counts.values, title='Образование: количество записей по годам окончания',
             labels={'x': 'Год окончания', 'y': 'Количество записей'})
fig.show()

По правилу IQR выбросами считаются все годы раньше 1995. Но у нас много педагогов старше 50 лет, и для них диплом 1980-х годов - это нормально. Значит, просто удалять старые годы нельзя.

Зато на графике шкала начинается с 1915 года, хотя столбиков там не видно: значит, есть единичные записи с очень странным годом. Поэтому проверим даты по смыслу: сравним год окончания с годом рождения педагога. Возраст возьмём из таблицы «Педагоги».

Ошибкой будем считать два случая:
- человек окончил учебное заведение раньше, чем ему исполнилось 14 лет;
- год окончания больше, чем год, за который собраны данные.

In [23]:
# возраст педагога в каждом году (в «Педагогах» много строк на одного человека, оставляем одну)
ages = df_ped[['id', 'year', 'Возраст']].drop_duplicates(subset=['id', 'year'])

df_edu = df_edu.merge(ages, on=['id', 'year'], how='left')
df_edu['Год рождения'] = df_edu['year'] - df_edu['Возраст']
df_edu['Возраст при окончании'] = df_edu['Дата окончания'] - df_edu['Год рождения']

too_early = df_edu['Возраст при окончании'] < 14
too_late = df_edu['Дата окончания'] > df_edu['year']

print('Окончили раньше 14 лет:', too_early.sum())
print('Год окончания из будущего:', too_late.sum())

Окончили раньше 14 лет: 309
Год окончания из будущего: 8


In [24]:
df_edu[too_early].sort_values('Дата окончания').head(5)

,id,Тип образования,Наименование учебного заведения,Дата окончания,Специализация,Квалификация,year,Возраст,Год рождения,Возраст при окончании
147740,8b152c1f29,"Высшее образование - специалитет, магистратура",Московский государственный гуманитарный универ...,1915,Педагогика и психология,Педагог-психолог,2023,44,1979,-64
69909,7f471e1525,Среднее профессиональное образование,Севское селькое профессионально-техническое уч...,1963,Техническое обслуживание и ремонт автомобильно...,Тракторист-машинист широкого профиля,2022,22,2000,-37
147827,7f471e1525,Среднее профессиональное образование,Севское селькое профессионально-техническое уч...,1963,Техническое обслуживание и ремонт автомобильно...,Тракторист-машинист широкого профиля,2023,23,2000,-37
19497,9db3b7e403,Высшее образование,Чечено-Ингушский государственный педагогически...,1967,Русский язык и литература,Учитель русского языка и литературы,2022,67,1955,12
93624,9db3b7e403,Высшее образование,Чечено-Ингушский государственный педагогически...,1967,Русский язык и литература,Учитель русского языка и литературы,2023,68,1955,12


Это явные ошибки ввода: например, педагог 1979 года рождения не мог окончить вуз в 1915 году.

**Решение:** строки не удаляем, потому что само образование у человека есть, неправильный только год. Настоящий год мы узнать не можем, поэтому заменяем его на ближайшее возможное значение:
- слишком ранний год → год, когда педагогу исполнилось 14 лет;
- год из будущего → год сбора данных.

Чтобы потом можно было найти исправленные строки, добавим столбец-отметку. Старые, но возможные даты не трогаем.

In [25]:
df_edu['Дата исправлена'] = too_early | too_late

df_edu.loc[too_early, 'Дата окончания'] = df_edu.loc[too_early, 'Год рождения'] + 14
df_edu.loc[too_late, 'Дата окончания'] = df_edu.loc[too_late, 'year']

# вспомогательные столбцы удаляем
df_edu = df_edu.drop(columns=['Возраст', 'Год рождения', 'Возраст при окончании'])

print('Исправлено дат:', df_edu['Дата исправлена'].sum())
print('Минимальный год:', df_edu['Дата окончания'].min())
print('Максимальный год:', df_edu['Дата окончания'].max())

Исправлено дат: 317
Минимальный год: 1959
Максимальный год: 2024


### 3.3. Школы - количество зданий и численность контингента

In [26]:
fig = px.box(df_school, x='Кол-во зданий', title='Школы: количество зданий', hover_data=['Школа', 'Округ'])
fig.show()

fig = px.box(df_school, x='Общая численность контингента', title='Школы: численность контингента', hover_data=['Школа', 'Округ'])
fig.show()

In [27]:
out_buildings

,Школа,Округ,Кол-во зданий,Общая численность контингента
29,46,Инновационный округ,37,6161
36,218,Западный округ,37,4085
88,418,Восточный округ,42,3800


In [28]:
out_pupils

,Школа,Округ,Кол-во зданий,Общая численность контингента
78,201,Южный округ,19,7124


В выбросы попали три школы с 37-42 зданиями и одна школа с 7124 учениками. Это большие образовательные комплексы, которые действительно существуют.

**Решение:** оставляем как есть. «Школы» - это справочник, в нём каждая строка - настоящая школа. Если удалить или изменить такие строки, директор крупной школы не найдёт её в системе или увидит неправильные цифры.

Чтобы большие и маленькие школы можно было сравнивать, добавим показатель «учеников на одно здание».

In [29]:
df_school['Учеников на здание'] = (df_school['Общая численность контингента'] / df_school['Кол-во зданий']).round()
df_school['Учеников на здание'].describe()

count     266.000000
mean      308.593985
std       159.149700
min        24.000000
25%       214.000000
50%       287.000000
75%       385.500000
max      1166.000000
Name: Учеников на здание, dtype: float64

### 3.4. Итог по выбросам

| Таблица | Столбец | Что нашли | Что сделали |
|---|---|---|---|
| ПК/ПП | Количество часов | длинные программы (до 260 часов) | оставили, добавили столбец с ограничением по 99-му перцентилю |
| Образование | Дата окончания | старые годы и невозможные даты | старые годы оставили, невозможные даты исправили и отметили |
| Школы | Кол-во зданий | три очень крупные школы | оставили |
| Школы | Численность контингента | одна очень крупная школа | оставили, добавили показатель «учеников на здание» |

## 4. Таблица для прототипа

В таблице «Педагоги» на одного человека приходится много строк: по одной на каждый предмет и параллель. Для графиков нам нужна одна строка на педагога в год. Как и на семинаре, оставляем первую запись для каждой пары `id` + `year`.

In [30]:
teachers = df_ped.drop_duplicates(subset=['id', 'year'])

# оставляем только нужные столбцы и даём им короткие названия
teachers = teachers[['id', 'year', 'ID образовательной организации', 'Пол', 'Возраст', 'Стаж педагогической работы',
                     'Квалификационная категория (педагогического работника)', 'Имеет классное руководство']]
teachers = teachers.rename(columns={
    'year': 'Год',
    'ID образовательной организации': 'Школа',
    'Стаж педагогической работы': 'Стаж',
    'Квалификационная категория (педагогического работника)': 'Категория',
    'Имеет классное руководство': 'Классное руководство',
})

print(teachers.shape)
teachers.head(3)

(54000, 8)


,id,Год,Школа,Пол,Возраст,Стаж,Категория,Классное руководство
0,f583c34a49,2022,478,Женский,35,1,NaN,Нет
4,08ff701e26,2022,523,Женский,54,24,NaN,Нет
10,b813df019b,2022,42,Женский,52,30,Высшая категория,Да


**Категория.** Посмотрим, какие значения встречаются.

In [31]:
teachers['Категория'].value_counts(dropna=False)

Категория
Высшая категория                      27071
Первая категория                      14527
NaN                                   12085
-                                       116
Вторая категория                        109
Высшая категория, Первая категория       89
Вторая категория, Первая категория        3
Name: count, dtype: int64

Значения записаны по-разному: есть пропуски, прочерки и по две категории сразу. Приведём всё к четырём вариантам. Если указаны две категории, берём старшую. Пропуск и прочерк означают, что категории нет.

In [32]:
def get_category(value):
    if pd.isna(value):
        return 'Нет категории'
    if 'Высшая' in value:
        return 'Высшая'
    if 'Первая' in value:
        return 'Первая'
    if 'Вторая' in value:
        return 'Вторая'
    return 'Нет категории'

teachers['Категория'] = teachers['Категория'].apply(get_category)
teachers['Категория'].value_counts()

Категория
Высшая           27160
Первая           14530
Нет категории    12201
Вторая             109
Name: count, dtype: int64

**Округ.** Добавляем округ из таблицы «Школы» по номеру школы.

In [33]:
teachers = teachers.merge(df_school[['Школа', 'Округ']], on='Школа', how='left')

print('Школ в таблице «Педагоги»:', teachers['Школа'].nunique())
print('Школ в таблице «Школы»:', len(df_school))
print('Строк без округа:', teachers['Округ'].isna().sum(), 'из', len(teachers))

Школ в таблице «Педагоги»: 478
Школ в таблице «Школы»: 266
Строк без округа: 1278 из 54000


В «Педагогах» организаций больше, чем в справочнике «Школы», поэтому у части педагогов округ не нашёлся. Таких строк немного (около 2 %). Удалять их не будем, чтобы не занижать общее число педагогов в городе, а запишем «Округ не указан».

In [34]:
teachers['Округ'] = teachers['Округ'].fillna('Округ не указан')

**Повышение квалификации.** Отметим, проходил ли педагог ПК и ПП в этом году: 1 - да, 0 - нет.

In [35]:
# педагоги, у которых есть ПК
with_pk = df_pk[df_pk['Тип программы'] == 'ПК'][['id', 'year']].drop_duplicates()
with_pk = with_pk.rename(columns={'year': 'Год'})
with_pk['Есть ПК'] = 1

# педагоги, у которых есть ПП
with_pp = df_pk[df_pk['Тип программы'] == 'ПП'][['id', 'year']].drop_duplicates()
with_pp = with_pp.rename(columns={'year': 'Год'})
with_pp['Есть ПП'] = 1

teachers = teachers.merge(with_pk, on=['id', 'Год'], how='left')
teachers = teachers.merge(with_pp, on=['id', 'Год'], how='left')

# у кого записи не нашлось, тот обучение не проходил
teachers['Есть ПК'] = teachers['Есть ПК'].fillna(0).astype(int)
teachers['Есть ПП'] = teachers['Есть ПП'].fillna(0).astype(int)

**Возрастная группа.** Делим возраст на группы, чтобы удобнее было сравнивать.

In [36]:
age_groups = ['до 25', '25-34', '35-44', '45-54', '55-64', '65+']
teachers['Возрастная группа'] = pd.cut(teachers['Возраст'], bins=[0, 24, 34, 44, 54, 64, 120], labels=age_groups)
teachers['Возрастная группа'] = teachers['Возрастная группа'].astype(str)

teachers.head(5)

,id,Год,Школа,Пол,Возраст,Стаж,Категория,Классное руководство,Округ,Есть ПК,Есть ПП,Возрастная группа
0,f583c34a49,2022,478,Женский,35,1,Нет категории,Нет,Восточный округ,1,0,35-44
1,08ff701e26,2022,523,Женский,54,24,Нет категории,Нет,Западный округ,1,0,45-54
2,b813df019b,2022,42,Женский,52,30,Высшая,Да,Северный округ,1,0,45-54
3,b484e1de85,2022,66,Женский,41,19,Высшая,Да,Северный округ,1,0,35-44
4,60f45bfe9e,2022,19,Женский,58,21,Высшая,Нет,Инновационный округ,1,0,55-64


In [37]:
print('Пропусков в таблице:', teachers.isna().sum().sum())
print()
print('Педагогов по годам:')
print(teachers['Год'].value_counts().sort_index())
print()
print('Доля педагогов с ПК по годам:')
print(teachers.groupby('Год')['Есть ПК'].mean().round(2))

Пропусков в таблице: 0

Педагогов по годам:
Год
2022    17833
2023    17799
2024    18368
Name: count, dtype: int64

Доля педагогов с ПК по годам:
Год
2022    0.81
2023    0.77
2024    0.73
Name: Есть ПК, dtype: float64


Получилась таблица, где одна строка - это один педагог в одном году, пропусков в ней нет. Сохраняем её в файл: прототип будет читать его, а не большие Excel-файлы, поэтому страницы будут открываться быстро.

In [38]:
teachers.to_csv('data_clean.csv', index=False, encoding='utf-8-sig')

## 5. Интерактивные визуализации

Графики строим в Plotly: при наведении мышки показываются точные значения, а нажатием на легенду можно скрыть ряд.

Для примера берём 2024 год и школу, в которой больше всего педагогов.

In [39]:
city = teachers[teachers['Год'] == 2024]

# школа с самым большим числом педагогов
school_id = city[city['Округ'] != 'Округ не указан']['Школа'].value_counts().index[0]
school = city[city['Школа'] == school_id]
school_name = 'Школа № ' + str(school_id)

print(school_name)
print('Педагогов в школе:', len(school))
print('Педагогов в городе:', len(city))

Школа № 280
Педагогов в школе: 221
Педагогов в городе: 18368


### 5.1. Требование директора школы

Директор хочет сравнить свою школу со средними значениями по городу. В школе и в городе разное число педагогов, поэтому сравниваем не количество, а доли в процентах.

**График 1. Квалификационные категории: школа и город**

In [40]:
categories = ['Высшая', 'Первая', 'Вторая', 'Нет категории']

school_pct = school['Категория'].value_counts(normalize=True) * 100
city_pct = city['Категория'].value_counts(normalize=True) * 100

compare_cat = pd.DataFrame({school_name: school_pct, 'Город': city_pct})
compare_cat = compare_cat.reindex(categories).fillna(0).round(1)
compare_cat

,Школа № 280,Город
Категория,,
Высшая,51.1,49.2
Первая,21.7,26.4
Вторая,0.0,0.2
Нет категории,27.1,24.3


In [41]:
fig = px.bar(compare_cat, barmode='group', title='Квалификационные категории: школа и город',
             labels={'value': 'Доля педагогов, %', 'Категория': 'Квалификационная категория', 'variable': ''},
             color_discrete_sequence=['#2a78d6', '#eb6834'])
fig.update_layout(template='plotly_white')
fig.show()

**Вывод.** В школе № 280 педагогов высшей категории чуть больше, чем в среднем по городу (51,1 % против 49,2 %). Зато первой категории меньше (21,7 % против 26,4 %), а педагогов без категории больше (27,1 % против 24,3 %). Директор видит, по какому показателю школа отстаёт, и может запланировать аттестацию для тех, у кого категории нет.

**График 2. Возрастная структура: школа и город**

In [42]:
school_pct = school['Возрастная группа'].value_counts(normalize=True) * 100
city_pct = city['Возрастная группа'].value_counts(normalize=True) * 100

compare_age = pd.DataFrame({school_name: school_pct, 'Город': city_pct})
compare_age = compare_age.reindex(age_groups).fillna(0).round(1)
compare_age

,Школа № 280,Город
Возрастная группа,,
до 25,9.5,7.6
25-34,17.2,19.8
35-44,22.6,20.1
45-54,24.0,28.0
55-64,20.4,19.7
65+,6.3,4.9


In [43]:
fig = px.bar(compare_age, barmode='group', title='Возрастная структура: школа и город',
             labels={'value': 'Доля педагогов, %', 'Возрастная группа': 'Возрастная группа', 'variable': ''},
             color_discrete_sequence=['#2a78d6', '#eb6834'])
fig.update_layout(template='plotly_white')
fig.show()

In [44]:
print('Средний возраст в школе:', round(school['Возраст'].mean(), 1))
print('Средний возраст в городе:', round(city['Возраст'].mean(), 1))
print('Средний стаж в школе:', round(school['Стаж'].mean(), 1))
print('Средний стаж в городе:', round(city['Стаж'].mean(), 1))

Средний возраст в школе: 44.5
Средний возраст в городе: 44.2
Средний стаж в школе: 19.1
Средний стаж в городе: 18.7


**Вывод.** Средний возраст и стаж в школе почти такие же, как в городе. Но по группам различия есть: в школе больше самых молодых (до 25 лет) и педагогов 35-44 лет, а группа 45-54 года заметно меньше городской. Педагогов старше 55 лет в школе чуть больше, чем в городе. По этому графику директор понимает, много ли в школе педагогов предпенсионного возраста и есть ли молодые, которые смогут их заменить.

### 5.2. Требование сотрудника городского органа управления

**График 3. Численность педагогов по округам**

In [45]:
okrug_counts = city['Округ'].value_counts().sort_values()
okrug_counts = okrug_counts.reset_index()
okrug_counts.columns = ['Округ', 'Педагогов']
okrug_counts

,Округ,Педагогов
0,Округ не указан,435
1,Инновационный округ,2682
2,Южный округ,3625
3,Западный округ,3704
4,Северный округ,3805
5,Восточный округ,4117


In [46]:
fig = px.bar(okrug_counts, x='Педагогов', y='Округ', orientation='h', text='Педагогов',
             title='Численность педагогов по округам', color_discrete_sequence=['#2a78d6'])
fig.update_layout(template='plotly_white', xaxis_title='Число педагогов', yaxis_title='')
fig.show()

**Вывод.** В 2024 году в городе 18 368 педагогов. Больше всего их в Восточном округе (4 117), меньше всего - в Инновационном (2 682), разница примерно в полтора раза. Ещё 435 педагогов работают в организациях, которых нет в справочнике школ. Сотрудник городского управления сразу видит размер округов и может учитывать его при распределении ресурсов.

**График 4. Возрастная структура по округам**

Для каждого округа считаем, какую долю составляет каждая возрастная группа.

In [47]:
age_by_okrug = pd.crosstab(city['Округ'], city['Возрастная группа'], normalize='index') * 100
age_by_okrug = age_by_okrug[age_groups].round(1)  # ставим группы по порядку от младших к старшим
age_by_okrug

Возрастная группа,до 25,25-34,35-44,45-54,55-64,65+
Округ,,,,,,
Восточный округ,7.8,20.1,20.5,27.8,19.4,4.3
Западный округ,6.7,18.8,20.0,29.8,20.6,4.1
Инновационный округ,7.6,19.2,18.2,26.1,22.1,6.8
Округ не указан,8.5,30.8,23.9,25.7,10.6,0.5
Северный округ,8.3,19.7,20.8,26.5,19.2,5.5
Южный округ,7.6,19.5,19.7,29.5,19.1,4.6


In [48]:
# оттенки синего: чем старше группа, тем темнее
blue_shades = ['#b7d3f6', '#86b6ef', '#5598e7', '#2a78d6', '#1c5cab', '#0d366b']

fig = px.bar(age_by_okrug, orientation='h', title='Возрастная структура педагогов по округам',
             labels={'value': 'Доля педагогов, %', 'Округ': '', 'Возрастная группа': ''},
             color_discrete_sequence=blue_shades)
fig.update_layout(template='plotly_white')
fig.show()

**Вывод.** В большинстве округов возрастная структура похожа, но Инновационный округ выделяется: там самая большая доля педагогов 55 лет и старше (22,1 % + 6,8 % = 28,9 %, в остальных округах около 24 %). Значит, в этом округе раньше всего понадобится замена уходящих на пенсию педагогов. В организациях без округа педагоги, наоборот, заметно моложе.

## Итоги

**Пропуски**
- В «ПК/ПП» и «Школах» пропусков нет.
- В «Образовании» текстовые столбцы заполнили значением «не указано», а год окончания - медианой по типу образования.

**Выбросы**
- Количество часов: длинные программы оставили, для графиков добавили столбец с ограничением.
- Дата окончания: невозможные даты исправили и отметили, старые даты оставили.
- Школы: крупные школы оставили, добавили показатель «учеников на здание».

**Для прототипа**
- Подготовили таблицу `data_clean.csv`: одна строка - один педагог в одном году.
- Построили по два интерактивных графика для директора школы и для сотрудника городского органа управления.

Дальше нужно перенести графики в приложение Streamlit, добавить фильтры и сделать скриншоты страниц.